<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">

LING 593A-011<br>
Fall 2025<br>
Davo Acevedo-Cardona

# Official Board Executive Changes Extraction Project Overview

## Task:
Extract structured executive movement data (appointments, promotions, departures) from Official Board Gmail alerts to build a dataset of executive changes across companies.

---

## Dataset:
- Gmail messages (Official Board alert emails)  

---

## What This Pipeline Does?

**Load:**  
Connects to Gmail API and retrieves Official Board alert emails.

**Extract:**  
Parses raw email content (HTML/plaintext) and converts it into clean text.

**Segment:**  
Identifies company-level blocks within each email and separates content by company.

**Split Sections:**  
Detects sections such as:
- Appointments  
- Changes  

**Parse Events:**  
Extracts structured information about executive movements, including:
- executive name  
- event type (join, promotion, departure, retirement)  
- old position  
- new position  

**Normalize:**  
Standardizes text fields by:
- cleaning names and titles  
- normalizing event types  

**Log Failures:**  
Tracks parsing failures for manual review.

---

## Libraries Used

**Packages:**
- pandas  
- re  
- base64  
- email  
- google-api-python-client  

---

## Pipeline

### Step 1: Gmail Connection / Data Loading
- Authenticate with Gmail API  
- Retrieve message IDs  
- Decode email bodies (HTML → text)  

---

### Step 2: Text Cleaning
- Remove HTML tags  
- Normalize whitespace  
- Extract readable plaintext  

---

### Step 3: Company Block Detection
- Identify company sections within each email  
- Split email into company-specific blocks  

---

### Step 4: Section Splitting
- Detect sections:
  - Appointments  
  - Changes  
- Separate content accordingly  

---

### Step 5: Event Extraction
- Parse executive movement statements  
- Identify event types:
  - joined / will join  
  - promoted / became  
  - left / will leave  
  - retired / will retire  

---

### Step 6: Field Extraction
Extract structured fields:
- company  
- executive_name  
- event_type  
- old_position  
- new_position  
- posted_date  
- email_date  

---

### Step 7: Failure Handling
- Log parsing errors  
- Store failed rows for manual review  

---

## Outputs

### CSV Files:
- `official_board_exec_changes.csv` → structured executive changes dataset  
- `official_board_parse_failures.csv` → failed parsing cases  

---

## Goal

Transform unstructured email alerts into a structured dataset capturing **executive movements across companies**, enabling:

- tracking leadership changes  
- integration with executive datasets  
- longitudinal corporate analysis  


# Imports

In [1]:
from __future__ import annotations

import base64
import csv
import json
import os
import re
import datetime as dt
from dataclasses import dataclass
from email.utils import parsedate_to_datetime
from typing import Optional, List, Dict, Tuple

from tqdm.auto import tqdm

# Class Configuration

In [2]:
@dataclass
class Config:
    gmail_query: str = 'from:alert@news.theofficialboard.com'
    max_messages: Optional[int] = None

    checkpoint_path: str = "official_board_checkpoint.jsonl"
    extracted_csv: str = "official_board_exec_changes.csv"
    failures_csv: str = "official_board_parse_failures.csv"


cfg = Config(
    gmail_query='from:alert@news.theofficialboard.com',
    max_messages=None,
)

# API Auth

In [3]:
def build_gmail_service(
    credentials_json_path: str = "credentials.json",
    token_path: str = "token.json"
):
    from googleapiclient.discovery import build
    from google_auth_oauthlib.flow import InstalledAppFlow
    from google.auth.transport.requests import Request
    from google.oauth2.credentials import Credentials

    SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                credentials_json_path, SCOPES
            )
            creds = flow.run_local_server(port=0)

        with open(token_path, "w", encoding="utf-8") as f:
            f.write(creds.to_json())

    return build("gmail", "v1", credentials=creds)

# Fetch

In [4]:
def list_message_ids(service, user_id: str, query: str, max_messages: Optional[int]):
    ids = []
    page_token = None

    while True:
        resp = service.users().messages().list(
            userId=user_id,
            q=query,
            pageToken=page_token,
            maxResults=500
        ).execute()

        for m in resp.get("messages", []) or []:
            ids.append(m["id"])
            if max_messages and len(ids) >= max_messages:
                return ids

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return ids


def get_message(service, user_id: str, message_id: str):
    return service.users().messages().get(
        userId=user_id,
        id=message_id,
        format="full"
    ).execute()


def get_header(headers, name: str) -> str:
    for h in headers:
        if h.get("name", "").lower() == name.lower():
            return h.get("value", "")
    return ""

# Email Body Decoding

In [5]:
def decode_b64url(data: str) -> str:
    if not data:
        return ""
    pad = len(data) % 4
    if pad:
        data += "=" * (4 - pad)
    return base64.urlsafe_b64decode(data).decode("utf-8", errors="replace")


def extract_plaintext(payload: dict) -> str:
    def walk(p):
        parts = [p]
        for c in p.get("parts", []) or []:
            parts.extend(walk(c))
        return parts

    for part in walk(payload):
        if part.get("mimeType") == "text/plain":
            return decode_b64url(part.get("body", {}).get("data", ""))

    return decode_b64url(payload.get("body", {}).get("data", ""))

# Checkpointing & CSV Writers

In [6]:
def load_processed_ids(path: str) -> set:
    seen = set()
    if not os.path.exists(path):
        return seen

    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                seen.add(json.loads(line)["message_id"])
            except Exception:
                pass

    return seen


def append_checkpoint(path: str, message_id: str):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps({"message_id": message_id}) + "\n")


def write_csv(path: str, rows: list, fields: list):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)

# Text Cleaning for Official Board Emails

In [7]:
# -----------------------------
# Industry dictionary used to split company+industry headers
# -----------------------------

INDUSTRY_TERMS = [
    "Aerospace",
    "Airlines",
    "Banking",
    "Biotechnology",
    "Broadcasting",
    "Business Services",
    "Casinos",
    "Communication & Sales",
    "Construction",
    "Consumer Electronics",
    "Financial Services",
    "Fund",
    "Furniture",
    "Holding",
    "Hotels",
    "Industrial Conglomerates",
    "Insurance",
    "Machinery",
    "Materials",
    "Pharmaceuticals",
    "Real Estate",
    "Recruiting",
    "Reinsurance",
    "Retail",
    "Semiconductors",
    "Software",
    "Telecommunications",
    "Video Games",
]

# Build regex pattern used in cleaning
import re

INDUSTRY_PATTERN = "|".join(
    sorted(map(re.escape, INDUSTRY_TERMS), key=len, reverse=True)
)

In [8]:
# -----------------------------
# Stronger Official Board cleaning
# -----------------------------

FOOTER_PATTERNS = [
    r"Your have reached the maximum level of information available.*",
    r"Click to create your signals on.*",
    r"Manage your existing alerts.*",
    r"Essentials — Monthly tips.*",
    r"Get Essentials.*",
    r"Org trends — Leadership changes.*",
    r"Get Org Trends.*",
    r"This alert is sent once a month by The Official Board\..*",
    r"Unsubscribe to this alert.*",
]

MONTH_LINE_RE = re.compile(
    r"^(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}",
    flags=re.IGNORECASE
)

def clean_official_board_text(body: str) -> str:
    text = body or ""

    # Normalize newlines first
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove URL-only bracket lines
    text = re.sub(r"\n?\[https?://[^\]]+\]\s*", "\n", text, flags=re.IGNORECASE)

    # Remove top header labels
    text = re.sub(r"THE OFFICIAL BOARD\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"/\s*MY ALERTS\s*", "\n", text, flags=re.IGNORECASE)

    # Put common markers on their own lines
    text = re.sub(r"(Appointments)", r"\n\1\n", text)
    text = re.sub(r"(Changes)", r"\n\1\n", text)
    text = re.sub(r"(Subsidiary of [^\n]+)", r"\n\1\n", text)
    text = re.sub(r"(View the new org chart of [^\n]+)", r"\n\1\n", text)
    text = re.sub(r"(/\s*Report an error)", r"\n\1\n", text)

    # Add newline after the alert date
    text = re.sub(
        r"((January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4})",
        r"\1\n",
        text,
        flags=re.IGNORECASE
    )

    # Split glued company+industry headers like "7-ElevenRetail"
    text = re.sub(
        rf"([A-Za-z0-9&.,'’/\- )]+?)({INDUSTRY_PATTERN})\b",
        r"\1\n\2",
        text
    )

    # Make common event starts begin on a new line
    text = re.sub(r"(?<!\n)([A-Z][A-Za-z0-9&.'’\- ]+\s+has joined the company as\b)", r"\n\1", text)
    text = re.sub(r"(?<!\n)([A-Z][A-Za-z0-9&.'’\- ]+\s+will join the company as\b)", r"\n\1", text)
    text = re.sub(r"(?<!\n)([A-Z][A-Za-z0-9&.'’\- ]+,\s+who was\b)", r"\n\1", text)
    text = re.sub(r"(?<!\n)([A-Z][A-Za-z0-9&.'’\- ]+\s+who was\b)", r"\n\1", text)
    text = re.sub(r"(?<!\n)([A-Z][A-Za-z0-9&.'’\- ]+,\s+who is\b)", r"\n\1", text)
    text = re.sub(r"(?<!\n)([A-Z][A-Za-z0-9&.'’\- ]+\s+who is\b)", r"\n\1", text)

    # Put newline after posted date
    text = re.sub(r"\(posted on [^)]+\)\.?", lambda m: m.group(0) + "\n", text, flags=re.IGNORECASE)

    # Remove weird spacing around punctuation
    text = re.sub(r"\s+,", ",", text)
    text = re.sub(r"\s+\.", ".", text)

    # Collapse spaces but keep newlines
    text = re.sub(r"[ \t]+", " ", text)

    # Remove footer sections
    for pat in FOOTER_PATTERNS:
        text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)

    # Clean extra newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"View\s+the\s+new\s+org\s+chart\s+of\s+", "\nView the new org chart of ", text, flags=re.IGNORECASE)
    text = re.sub(r"/\s*Report\s+an\s+error", "\n/ Report an error\n", text, flags=re.IGNORECASE)

    return text.strip()

# Split Email into Company Blocks

In [9]:
def split_company_blocks(text: str) -> List[Dict[str, str]]:
    blocks = []

    # Normalize wrapped "View the new org chart..." and "/ Report an error"
    text = re.sub(r"View\s+the\s+new\s+org\s+chart\s+of\s+", "View the new org chart of ", text, flags=re.IGNORECASE)
    text = re.sub(r"/\s*Report\s+an\s+error", "/ Report an error", text, flags=re.IGNORECASE)

    # Split on the org-chart marker
    raw_blocks = re.split(r"\nView the new org chart of .*?(?=\n|$)", text)

    for raw in raw_blocks:
        raw = raw.strip()
        if not raw:
            continue

        lines = [ln.strip() for ln in raw.splitlines() if ln.strip()]
        if len(lines) < 3:
            continue

        # Remove noise at the top
        while lines and (
            lines[0].upper().startswith("THE OFFICIAL BOARD")
            or lines[0].lower().startswith("forwarded message")
            or lines[0].lower().startswith("from:")
            or lines[0].lower().startswith("date:")
            or lines[0].lower().startswith("subject:")
            or lines[0].lower().startswith("to:")
            or MONTH_LINE_RE.match(lines[0])
            or lines[0] == "/ Report an error"
        ):
            lines = lines[1:]

        # Remove noise at the bottom
        while lines and lines[-1] == "/ Report an error":
            lines = lines[:-1]

        if len(lines) < 3:
            continue

        company = lines[0]
        industry = lines[1]

        # Only keep real company blocks
        if industry not in INDUSTRY_TERMS:
            continue

        idx = 2
        subsidiary_of = ""

        if idx < len(lines) and lines[idx].startswith("Subsidiary of "):
            subsidiary_of = lines[idx].replace("Subsidiary of ", "").strip()
            idx += 1

        block_body = "\n".join(lines[idx:]).strip()

        if "Appointments" not in block_body and "Changes" not in block_body:
            continue

        blocks.append({
            "company": company,
            "industry": industry,
            "subsidiary_of": subsidiary_of,
            "block_body": block_body
        })

    return blocks

# Split a Company Block into Sections

In [10]:
def split_sections(block_body: str) -> List[Tuple[str, List[str]]]:
    """
    Returns:
    [
        ("Appointments", [event1, event2, ...]),
        ("Changes", [event1, event2, ...]),
    ]

    Rebuilds events by section and splits on '(posted on ...)' without lookbehind.
    """
    text = block_body.strip()

    # Normalize spaces a bit
    text = re.sub(r"[ \t]+", " ", text)

    # Fix glued event starts after posted date
    text = re.sub(
        r"(\(posted on [^)]+\)\.)([A-Z])",
        r"\1\n\2",
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(
        r"(\(posted on [^)]+\))([A-Z])",
        r"\1\n\2",
        text,
        flags=re.IGNORECASE
    )

    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]

    sections = []
    current_section = None
    current_lines = []

    def flush_section():
        nonlocal current_section, current_lines, sections
        if current_section is None:
            return

        section_text = "\n".join(current_lines).strip()

        # Split AFTER posted-date markers by inserting a sentinel
        section_text = re.sub(
            r"(\(posted on [^)]+\)\.?)",
            r"\1<<<EVENT_SPLIT>>>",
            section_text,
            flags=re.IGNORECASE
        )

        raw_events = section_text.split("<<<EVENT_SPLIT>>>")

        events = []
        for ev in raw_events:
            ev = ev.strip()
            if not ev:
                continue
            ev = re.sub(r"\s+", " ", ev).strip()
            events.append(ev)

        sections.append((current_section, events))
        current_lines = []

    for line in lines:
        if line in {"Appointments", "Changes"}:
            flush_section()
            current_section = line
            current_lines = []
        else:
            if current_section is not None:
                current_lines.append(line)

    flush_section()
    return sections

# Date Helpers

In [11]:
POSTED_DATE_RE = re.compile(
    r"\(posted on ([^)]+)\)",
    flags=re.IGNORECASE
)

EFFECTIVE_DATE_RE = re.compile(
    r"\bon ([A-Z][a-z]{2,}\.? \d{1,2}, \d{4})\b"
)

def extract_posted_date(text: str) -> str:
    m = POSTED_DATE_RE.search(text)
    return m.group(1).strip() if m else ""


def extract_effective_date(text: str) -> str:
    m = EFFECTIVE_DATE_RE.search(text)
    return m.group(1).strip() if m else ""

# Event Parsing Logic

In [12]:
def clean_event_text(line: str) -> str:
    line = line.strip()
    line = re.sub(r"\s+", " ", line)
    line = re.sub(r"\s+,", ",", line)
    line = re.sub(r"\s+\.", ".", line)
    line = re.sub(r"\(\s*posted on", "(posted on", line, flags=re.IGNORECASE)
    return line


def remove_posted_suffix(line: str) -> str:
    return re.sub(r"\s*\(posted on [^)]+\)\.?$", "", line, flags=re.IGNORECASE).strip()


def parse_event_line(line: str) -> Dict[str, str]:
    """
    Parse one executive event line into structured fields.
    Returns a dict with:
    event_type, executive_name, old_position, new_position, effective_date, posted_date, raw_event_text
    """
    raw_event_text = clean_event_text(line)
    posted_date = extract_posted_date(raw_event_text)
    effective_date = extract_effective_date(raw_event_text)
    core = remove_posted_suffix(raw_event_text)

    result = {
        "event_type": "",
        "executive_name": "",
        "old_position": "",
        "new_position": "",
        "effective_date": effective_date,
        "posted_date": posted_date,
        "raw_event_text": raw_event_text
    }

    # 1) has joined the company as ...
    m = re.match(
        r"^(.*?)(?: has joined the company as )(.+?)\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "joined"
        result["executive_name"] = m.group(1).strip(" ,")
        result["new_position"] = m.group(2).strip(" ,.")
        return result

    # 2) will join the company as ... on DATE
    m = re.match(
        r"^(.*?)(?: will join the company as )(.+?)(?: on .+?)?\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "will_join"
        result["executive_name"] = m.group(1).strip(" ,")
        result["new_position"] = m.group(2).strip(" ,.")
        return result

    # 3) who was X, is promoted to Y
    m = re.match(
        r"^(.*?), who was (.*?), is promoted to (.+?)\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "promoted"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        result["new_position"] = m.group(3).strip(" ,.")
        return result

    # 4) who was X, becomes Y
    m = re.match(
        r"^(.*?)(?: who was )(.*?)(?:, becomes )(.+?)\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "became"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        result["new_position"] = m.group(3).strip(" ,.")
        return result

    # 5) who is X, will be promoted to Y on DATE
    m = re.match(
        r"^(.*?)(?: who is )(.*?)(?:, will be promoted to )(.+?)(?: on .+?)?\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "will_be_promoted"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        result["new_position"] = m.group(3).strip(" ,.")
        return result

    # 6) who is X, will become Y on DATE
    m = re.match(
        r"^(.*?)(?: who is )(.*?)(?:, will become )(.+?)(?: on .+?)?\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "will_become"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        result["new_position"] = m.group(3).strip(" ,.")
        return result

    # 7) X, POSITION, has left the company
    m = re.match(
        r"^(.*?), (.*?), has left the company\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "left"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        return result

    # 8) X, POSITION, will leave the company on DATE
    m = re.match(
        r"^(.*?), (.*?), will leave the company(?: on .+?)?\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "will_leave"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        return result

    # 9) X, POSITION, will retire on DATE
    m = re.match(
        r"^(.*?), (.*?), will retire(?: on .+?)?\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "will_retire"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        return result

    # 10) X, who is POSITION will soon retire
    m = re.match(
        r"^(.*?), who is (.*?)(?: will soon retire)\.?$",
        core,
        flags=re.IGNORECASE
    )
    if m:
        result["event_type"] = "will_retire"
        result["executive_name"] = m.group(1).strip(" ,")
        result["old_position"] = m.group(2).strip(" ,")
        return result

    # fallback
    result["event_type"] = "unparsed"
    return result

# Parse a Full Company Block

In [13]:
def parse_company_block(
    block: Dict[str, str],
    message_id: str,
    email_date: str,
    email_subject: str
) -> Tuple[List[Dict[str, str]], List[Dict[str, str]]]:
    """
    Returns:
    extracted_rows, failure_rows
    """
    extracted_rows = []
    failure_rows = []

    company = block["company"]
    industry = block["industry"]
    subsidiary_of = block["subsidiary_of"]
    block_body = block["block_body"]

    sections = split_sections(block_body)

    for section_type, lines in sections:
        for line in lines:
            parsed = parse_event_line(line)

            row = {
                "message_id": message_id,
                "email_date": email_date,
                "email_subject": email_subject,
                "company": company,
                "industry": industry,
                "subsidiary_of": subsidiary_of,
                "section_type": section_type,
                "event_type": parsed["event_type"],
                "executive_name": parsed["executive_name"],
                "old_position": parsed["old_position"],
                "new_position": parsed["new_position"],
                "effective_date": parsed["effective_date"],
                "posted_date": parsed["posted_date"],
                "raw_event_text": parsed["raw_event_text"],
            }

            extracted_rows.append(row)

            if parsed["event_type"] == "unparsed":
                failure_rows.append(row.copy())

    return extracted_rows, failure_rows

# Main Extraction Loop

In [14]:
def run(cfg: Config):
    service = build_gmail_service()

    processed = load_processed_ids(cfg.checkpoint_path)

    msg_ids = list_message_ids(
        service,
        "me",
        cfg.gmail_query,
        cfg.max_messages
    )

    to_process = [mid for mid in msg_ids if mid not in processed]

    print(f"Found {len(msg_ids)} messages.")
    print(f"Skipping {len(msg_ids) - len(to_process)} already processed.")
    print(f"Processing {len(to_process)} messages.\n")

    extracted = []
    failures = []

    pbar = tqdm(
        to_process,
        total=len(to_process),
        desc="Processing Official Board emails",
        unit="email",
        dynamic_ncols=True
    )

    for mid in pbar:
        msg = get_message(service, "me", mid)
        payload = msg.get("payload", {})
        headers = payload.get("headers", [])

        subject = get_header(headers, "Subject")
        sender = get_header(headers, "From")
        date_raw = get_header(headers, "Date")

        try:
            msg_dt = parsedate_to_datetime(date_raw)
        except Exception:
            internal_ms = int(msg.get("internalDate", "0"))
            msg_dt = dt.datetime.fromtimestamp(
                internal_ms / 1000.0,
                tz=dt.timezone.utc
            )

        body = extract_plaintext(payload)
        body = clean_official_board_text(body)

        blocks = split_company_blocks(body)

        for block in blocks:
            rows, bad_rows = parse_company_block(
                block=block,
                message_id=mid,
                email_date=msg_dt.isoformat(),
                email_subject=subject
            )
            extracted.extend(rows)
            failures.extend(bad_rows)

        append_checkpoint(cfg.checkpoint_path, mid)

        pbar.set_postfix(
            rows=len(extracted),
            failures=len(failures)
        )

    write_csv(
        cfg.extracted_csv,
        extracted,
        [
            "message_id",
            "email_date",
            "email_subject",
            "company",
            "industry",
            "subsidiary_of",
            "section_type",
            "event_type",
            "executive_name",
            "old_position",
            "new_position",
            "effective_date",
            "posted_date",
            "raw_event_text",
        ]
    )

    write_csv(
        cfg.failures_csv,
        failures,
        [
            "message_id",
            "email_date",
            "email_subject",
            "company",
            "industry",
            "subsidiary_of",
            "section_type",
            "event_type",
            "executive_name",
            "old_position",
            "new_position",
            "effective_date",
            "posted_date",
            "raw_event_text",
        ]
    )

    print("\nRun complete.")
    print(f"Total extracted rows: {len(extracted)}")
    print(f"Total parse failures: {len(failures)}")

# Run

In [15]:
run(cfg)

Found 175 messages.
Skipping 0 already processed.
Processing 175 messages.



Processing Official Board emails:   0%|                                                     | 0/175 [00:00<?, …


Run complete.
Total extracted rows: 8562
Total parse failures: 513


# Checks

In [16]:
import pandas as pd

df = pd.read_csv(cfg.extracted_csv)
fail = pd.read_csv(cfg.failures_csv)

print("Extracted rows:", len(df))
print("Unique companies:", df["company"].nunique())
print("\nEvent type counts:")
print(df["event_type"].value_counts(dropna=False).head(20))

print("\nParse failures:", len(fail))
display(fail.head(20))

Extracted rows: 8562
Unique companies: 2825

Event type counts:
event_type
left                1935
promoted            1719
became              1322
joined               942
will_become          614
will_be_promoted     561
unparsed             513
will_leave           473
will_retire          271
will_join            212
Name: count, dtype: int64

Parse failures: 513


,message_id,email_date,email_subject,company,industry,subsidiary_of,section_type,event_type,executive_name,old_position,new_position,effective_date,posted_date,raw_event_text
0,19ca95a2e88dfe4b,2026-03-01T00:23:16+00:00,100 News on S&P 500,Ametek,Consumer Electronics,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,February 10,"Thomas Montgomery, Senior Advisor, is retiring..."
1,19c192e837525e27,2026-02-01T00:23:18+00:00,100 News on Russell 3000,Absa,Banking,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,January 19,"Luisa Diogo, Independent Non-Executive Directo..."
2,19c192e826b30eff,2026-02-01T00:23:18+00:00,100 News on Global Fortune 500,Bouygues,Construction,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,January 5,"Pascal Grangé, Deputy Chief Executive Officer ..."
3,19c192819cb81da8,2026-02-01T00:23:18+00:00,100 News on S&P 500,Brown & Brown,Insurance,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,January 27,"Robert Mathis, Executive Vice President, Chief..."
4,1860ba7fa973c4ca,2023-02-01T01:23:08+00:00,1125 News on Russell 3000,FleetCor,Financial Services,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,January 30,"within the existing contacts:Mark Johnson, Dir..."
5,1860ba7fa973c4ca,2023-02-01T01:23:08+00:00,1125 News on Russell 3000,Berkshire Bank,Banking,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,January 18,"Georgia Melas, Senior Executive Vice President..."
6,1860ba7fa973c4ca,2023-02-01T01:23:08+00:00,1125 News on Russell 3000,Entegris,Semiconductors,NaN,Changes,unparsed,NaN,NaN,NaN,NaN,January 24,"within the existing contacts:Gregory Graves, w..."
7,1860ba7fa973c4ca,2023-02-01T01:23:08+00:00,1125 News on Russell 3000,Textron Systems,Aerospace,Textron,Changes,unparsed,NaN,NaN,NaN,NaN,January 24,Tom Hammoor is promoted to President and Chief...
8,1860ba7fa973c4ca,2023-02-01T01:23:08+00:00,1125 News on Russell 3000,LaSalle Europe,Real Estate,LaSalle,Changes,unparsed,NaN,NaN,NaN,NaN,January 20,"within the existing contacts: Katie Hynard, wh..."
9,1860ba7fa973c4ca,2023-02-01T01:23:08+00:00,1125 News on Russell 3000,Prudential Thailand,Insurance,Prudential Asia,Changes,unparsed,NaN,NaN,NaN,NaN,January 18,"Robin Spencer who was, becomes (posted on Janu..."
